1. softmax回归的简洁实现<br>
通过深度学习框架的高级API能够使实现softmax回归变得更加容易

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

- softmax 回归的输出层是一个全连接层

In [3]:
# pytorch不会隐式地调整输入的形状，因此我们需要在这里调整输入的形状
# 我们定义展平层（flatten），在线性层前调整网络输入的形状
# flatten就是将任何维度的展开为2d
net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)
net.apply(init_weights);

In [4]:
net = net.to(device)

- 在交叉熵损失函数中传递未归一化的预测，并且同时计算softmax及其对数

In [5]:
loss = nn.CrossEntropyLoss()

- 使用学习率为0.1的小批量随机梯度下降作为优化算法

In [6]:
trainer = torch.optim.SGD(net.parameters(), lr=0.1)

- 调用之前定义的训练训练函数来训练模型

In [7]:
def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(dim=1)
    cmp = (y_hat.type(y.dtype) == y)
    return float(cmp.type(torch.float32).sum())

def evaluate_accuracy(net, data_iter):
    net.eval()
    device = next(net.parameters()).device
    metric_correct, metric_total = 0.0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            metric_correct += accuracy(y_hat, y)
            metric_total += y.numel()
    return metric_correct / metric_total

def train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer):
    device = next(net.parameters()).device
    for epoch in range(num_epochs):
        net.train()
        total_loss, total_correct, total_num = 0.0, 0.0, 0
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            trainer.zero_grad()
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            trainer.step()

            total_loss += float(l) * y.numel()
            total_correct += accuracy(y_hat, y)
            total_num += y.numel()

        train_loss = total_loss / total_num
        train_acc = total_correct / total_num
        test_acc = evaluate_accuracy(net, test_iter)
        print(f"epoch {epoch+1}: loss={train_loss:.4f}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}")

In [9]:
num_epochs = 10
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

epoch 1: loss=0.4431, train_acc=0.8495, test_acc=0.8345
epoch 2: loss=0.4390, train_acc=0.8509, test_acc=0.8315
epoch 3: loss=0.4365, train_acc=0.8516, test_acc=0.8365
epoch 4: loss=0.4331, train_acc=0.8518, test_acc=0.8359
epoch 5: loss=0.4298, train_acc=0.8538, test_acc=0.8328
epoch 6: loss=0.4284, train_acc=0.8543, test_acc=0.8395
epoch 7: loss=0.4264, train_acc=0.8539, test_acc=0.8381
epoch 8: loss=0.4243, train_acc=0.8555, test_acc=0.8321
epoch 9: loss=0.4222, train_acc=0.8555, test_acc=0.8322
epoch 10: loss=0.4201, train_acc=0.8566, test_acc=0.8381
